# Lab 7: Planning, Reflection, and Verification (Google ADK)

**Note:** this lab session is graded, please complete all exercises and submit it to canvas.

A reactive agent chooses its next action from the latest observation alone. That is often sufficient for short, simple tasks, but longer or more failure-prone tasks may benefit from an explicit plan, delegation to specialized agents, and a review step that can send work back for revision. These mechanisms add cost and complexity, so they should earn their place rather than being added for their own sake.

In this lab, we will build a synthetic course-scheduling assistant twice, in two different styles. First, a single reactive agent that answers a scheduling question by calling tools directly, one step at a time. Second, an orchestrator agent that delegates to three specialists -- one that plans before anything executes, one that carries out the plan using the same tools, and one that reviews the resulting draft and can send it back for another pass. Every run is bounded by explicit turn, tool-call, token, and wall-clock limits, and every tool call, observation, and stated reasoning step is printed as it happens. The default model is `gpt-5-nano`. All deadlines and policies are synthetic and read-only; nothing is submitted, scheduled, approved, or changed externally.

## References and further reading

1. [Yao et al. (2022), *ReAct: Synergizing Reasoning and Acting in Language Models*](https://arxiv.org/abs/2210.03629) — the reason/act/observe loop the Part 1 agent (and each sub-agent in Part 2) implements.
2. [Shinn et al. (2023), *Reflexion: Language Agents with Verbal Reinforcement Learning*](https://arxiv.org/abs/2303.11366) — verbal feedback from one attempt conditioning the next; the mechanism the reflect-then-react handoff implements.
3. [Roitman, H. (2026), *The Hitchhiker's Guide to Agentic AI: From Foundations to Systems*](https://arxiv.org/abs/2606.24937) — Sections 18.5.1–18.5.2 and 19.5–19.8 for planning, execution, reflection, and verification; Sections 18.4–18.7 and Chapter 19 for bounded execution.
4. [Anthropic, *Building Effective Agents*](https://www.anthropic.com/research/building-effective-agents) — workflows versus autonomous agents, and orchestrator/worker delegation.
5. [Google ADK documentation](https://google.github.io/adk-docs/)
6. [Google ADK on GitHub](https://github.com/google/adk-python)

Today, we will build both agents using Google's Agent Development Kit (ADK) rather than hand-writing the tool-calling loop, state tracking, and agent-to-agent handoffs ourselves. ADK abstracts away a basic agentic harness for us: it implements the propose-execute-observe loop, conversation state and sessions, and delegation between agents, so the code below is almost entirely *what* each agent should do, not *how* a loop or a handoff is implemented.

## What is Google ADK?

Google's Agent Development Kit (ADK) is Google's open-source, model-agnostic Python framework for building and running LLM agents. You define an `Agent` with a name, an instruction, and a list of plain Python functions as `tools`; ADK turns each function's signature and docstring into a tool definition, lets the model decide which tool to call and with what arguments, executes the function for you, and feeds the result back to the model — repeating that propose-execute-observe cycle until the model produces a final answer. That loop is exactly the ReAct pattern (reason, act, observe) from Week 5. An `Agent` can also be given `sub_agents`: other agents it can delegate a turn to, by name, through ADK's built-in transfer mechanism -- this is how a single **orchestrator** agent coordinates several **specialist** agents without any Python control-flow code, which is what Part 2 below uses. ADK is model-agnostic: it can run Gemini directly, or any OpenAI-compatible model — including this course's `gpt-5-nano` — through its built-in `LiteLlm` adapter, so no new API key or provider is needed. An ADK `Runner` drives one agent (and, when it delegates, its sub-agents) through one conversation `Session`, and every step is exposed as an `Event` with an `author` field, which is what lets us print and inspect who is acting, what they are reasoning, and how much budget they have used as the run proceeds.

Both parts below carry explicit, Python-enforced budgets — a turn limit, a tool-call limit, a (best-effort) token limit, and a wall-clock timeout — the four independent bounds Week 5 argues a loop needs, because each one protects against a different failure (a few turns can still hide many tool calls; cheap calls can still run long; token cost is its own dimension). None of these four live only in a prompt: `run_and_show` checks and enforces all of them itself, every run, and prints a budget summary at the end regardless of how the run finished.

Nothing here uses Pydantic schemas or typed plan objects — plans and verdicts are just text. See the Takeaways at the end for what this buys you versus what it costs, compared to a version where Python enforces the whole loop directly.

**On "reasoning":** what is printed below is each agent's own *visible, stated rationale* -- the one-line justification its instruction requires it to write before acting -- not a hidden internal chain-of-thought. That is a deliberate choice, not a limitation: Week 5 is explicit that evaluation and debugging need an agent's observable decisions (tool name, arguments, result, stated rationale), not private reasoning tokens, and OpenAI reasoning models like `gpt-5-nano` do not expose the latter through this kind of API regardless of framework. `run_and_show` also checks for a `thought` flag on each content part in case the underlying model/framework surfaces one, and labels it distinctly if so.

> **Before running:** confirm `Agent`, `sub_agents`, `Runner`, `InMemorySessionService`, `LiteLlm`, and the `Event` fields used below (`author`, `function_call`, `function_response`, `text`, `thought`, `usage_metadata`, `is_final_response()`) against your installed `google-adk` version (`pip show google-adk`). Token accounting in particular is best-effort: confirm `event.usage_metadata.total_token_count` is actually populated when routing through `LiteLlm`, or treat the printed token counts as approximate.


## 0. Install and import

In [24]:
%pip install -q --upgrade google-adk litellm openai


In [25]:
import os
import time
from datetime import date, timedelta

# Change this line
from google.adk.tools import AgentTool
from google.adk import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

import litellm
litellm.suppress_debug_info = True

#MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")
try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = os.getenv("OPENAI_API_KEY")

MODEL = os.getenv("OPENAI_MODEL", "@30875-fall26-openai/gpt-5-nano")
base_url="https://api.portkey.ai/v1"

print("Model:", MODEL)


Model: @30875-fall26-openai/gpt-5-nano


## 1. Environment: synthetic deadlines and policies, as plain Python functions

These three functions *are* the tools — no wrapper classes, no schemas. ADK reads each
function's signature and docstring to figure out what arguments the model may supply.


In [26]:
DEADLINES = {
    "project_1_presentation_and_report": "2026-11-05",
    "project_2_presentation": "2026-12-03",
    "project_2_report": "2026-12-10",
}
POLICIES = {
    "late_work": ("A total two-day grace allowance applies only to eligible assignments. "
                  "It does not automatically apply to presentations or team milestones."),
    "project_2": ("Project 2 requires a presentation and demonstration, evaluation evidence, "
                 "an architecture defense, a security and human-oversight analysis, and a final report."),
}

def lookup_deadline(item: str) -> dict:
    """Look up the deadline for a course deliverable.

    Args:
        item: One of project_1_presentation_and_report, project_2_presentation, project_2_report.
    """
    if item not in DEADLINES:
        return {"ok": False, "error": "unknown_deadline", "item": item}
    return {"ok": True, "item": item, "deadline": DEADLINES[item]}

def lookup_policy(topic: str) -> dict:
    """Look up a course policy by topic.

    Args:
        topic: One of late_work, project_2.
    """
    if topic not in POLICIES:
        return {"ok": False, "error": "unknown_policy", "topic": topic}
    return {"ok": True, "topic": topic, "policy": POLICIES[topic]}

def calculate_offset(iso_date: str, days: int) -> dict:
    """Shift an ISO date by a signed number of days, from -30 to 30.

    Args:
        iso_date: Source date in YYYY-MM-DD form.
        days: Signed day offset to add to iso_date.
    """
    if not -30 <= days <= 30:
        return {"ok": False, "error": "offset_out_of_range"}
    try:
        result = date.fromisoformat(iso_date) + timedelta(days=days)
    except ValueError:
        return {"ok": False, "error": "invalid_date"}
    return {"ok": True, "source_date": iso_date, "days": days, "result_date": result.isoformat()}


## 2. Configure API access and the shared ADK model

Reads `OPENAI_API_KEY` from the environment, same as the rest of the course. Never place a
key in source or saved outputs. `run_and_show` is the one helper every agent call below uses,
and it is where every budget is defined and enforced.


In [27]:

ADK_MODEL = LiteLlm(
    model=MODEL,
    api_key=api_key,
    custom_llm_provider = "openai",
    api_base=base_url,
)

APP_NAME = "lab7_react_plan_reflect"
USER_ID = "student"
session_service = InMemorySessionService()

async def run_and_show(
    agent: Agent,
    message: str,
    label: str = "",
    max_turns: int = 20,
    max_tool_calls: int = 8,
    max_tokens: int | None = 20_000,
    max_seconds: float = 60.0,
) -> tuple[str, bool]:
    """Run one turn against an ADK agent (and, if it delegates, its sub-agents), print a
    live transcript, and enforce four independent, Python-checked budgets every single
    run -- a turn limit, a tool-call limit, a (best-effort) token limit, and a wall-clock
    timeout. These are the same four bound types Week 5 names as necessary (and
    individually insufficient on their own): a few turns can still hide many tool calls,
    cheap calls can still run long, and token cost is its own dimension. None of these
    rely on any agent's instructions -- they are checked here, after every event,
    regardless of what the agent (or orchestrator) wanted to do next.

    Returns (final_text, any_tool_reported_failure).
    """
    if label:
        print(f"\n{'='*22} {label} {'='*22}")
    runner = Runner(agent=agent, app_name=APP_NAME, session_service=session_service)
    session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID)
    content = types.Content(role="user", parts=[types.Part(text=message)])
    final_text, failed = "", False
    reasoning_step = 0
    turns, tool_calls, tokens_used = 0, 0, 0
    started = time.perf_counter()
    stop_reason = None
    async for event in runner.run_async(user_id=USER_ID, session_id=session.id, new_message=content):
        turns += 1
        usage = getattr(event, "usage_metadata", None)
        if usage is not None:
            tokens_used += getattr(usage, "total_token_count", 0) or 0
        author = getattr(event, "author", "?")
        for part in (event.content.parts if event.content else []):
            if getattr(part, "function_call", None):
                tool_calls += 1
                reasoning_step += 1
                call = part.function_call
                print(f"\n[loop {reasoning_step}] [{author}] Action:      {call.name}({call.args})")
            elif getattr(part, "function_response", None):
                resp = part.function_response
                print(f"[loop {reasoning_step}] [{author}] Observation: {resp.response}")
                if isinstance(resp.response, dict) and resp.response.get("ok") is False:
                    failed = True
            elif getattr(part, "text", None) and part.text.strip():
                is_thought = bool(getattr(part, "thought", False))
                if event.is_final_response():
                    print(f"\n[{author}] Final answer: {part.text.strip()}")
                else:
                    reasoning_step += 1
                    tag = "Thought (native reasoning)" if is_thought else "Reasoning"
                    print(f"\n[loop {reasoning_step}] [{author}] {tag}:   {part.text.strip()}")
        if event.is_final_response() and event.content and event.content.parts:
            final_text = event.content.parts[0].text or final_text
        elapsed = time.perf_counter() - started
        if turns >= max_turns:
            stop_reason = f"max_turns={max_turns} reached"
        elif tool_calls >= max_tool_calls:
            stop_reason = f"max_tool_calls={max_tool_calls} reached"
        elif max_tokens is not None and tokens_used >= max_tokens:
            stop_reason = f"max_tokens={max_tokens} reached ({tokens_used} used)"
        elif elapsed >= max_seconds:
            stop_reason = f"max_seconds={max_seconds} reached ({elapsed:.1f}s elapsed)"
        if stop_reason:
            print(f"\n!!! BUDGET EXHAUSTED: {stop_reason} -- stopping here regardless of what the agent wanted to do next.")
            break
    elapsed = time.perf_counter() - started
    status = f"STOPPED ({stop_reason})" if stop_reason else "completed within budget"
    print(f"\n[budget] turns={turns}/{max_turns}  tool_calls={tool_calls}/{max_tool_calls}  "
          f"tokens~={tokens_used}/{max_tokens}  elapsed={elapsed:.1f}s/{max_seconds}s  -- {status}")
    return final_text, failed


## Part 1: A basic ReAct agent

One agent, three tools, no plan, and no sub-agents. The budget is passed explicitly at the
call site below, not left to a default. Watch the live transcript show each reasoning step,
tool call, and observation as they happen, followed by the final answer and the budget
summary line.


In [28]:
simple_react_agent = Agent(
    name="simple_react_agent",
    model=ADK_MODEL,
    instruction=(
        "You help students with course scheduling questions. Use the available tools to "
        "look up deadlines, policies, or compute dates -- do not guess or invent values. "
        "Before every tool call, and before your final answer, write one line starting with "
        "'Thought:' explaining your reasoning -- never skip this. Once you have everything "
        "you need, answer the question directly."
    ),
    tools=[lookup_deadline, lookup_policy, calculate_offset],
)

# A single agent with three tools rarely needs more than a few turns or calls --
# budgets are deliberately tight here so a runaway loop is obvious if it happens.
answer, _ = await run_and_show(
    simple_react_agent,
    "When is the Project 2 presentation, what is the checkpoint date seven days before it, "
    "and what does the Project 2 policy require?",
    label="REACT AGENT",
    max_turns=100,
    max_tool_calls=40,
    max_tokens=8_0000,
    max_seconds=300,
)



====================== REACT AGENT ======================

[loop 1] [simple_react_agent] Reasoning:   Thought: I will fetch the Project 2 presentation deadline and the Project 2 policy in parallel to save time.

[loop 2] [simple_react_agent] Action:      lookup_deadline({'item': 'project_2_presentation'})

[loop 3] [simple_react_agent] Action:      lookup_policy({'topic': 'project_2'})
[loop 3] [simple_react_agent] Observation: {'ok': True, 'item': 'project_2_presentation', 'deadline': '2026-12-03'}
[loop 3] [simple_react_agent] Observation: {'ok': True, 'topic': 'project_2', 'policy': 'Project 2 requires a presentation and demonstration, evaluation evidence, an architecture defense, a security and human-oversight analysis, and a final report.'}

[loop 4] [simple_react_agent] Reasoning:   Thought: I will compute the date seven days before the December 3, 2026 deadline using the date offset tool.

[loop 5] [simple_react_agent] Action:      calculate_offset({'iso_date': '2026-12-03', 'd

## Part 2: A plan-react-reflect orchestrator with sub-agents

**How this differs from Part 1:** Part 1 is one agent making its own turn-by-turn calls. Here there are four agents: three specialists (`planner_agent`, `react_agent`, `reflect_agent`) and one `orchestrator_agent` whose only job is to delegate between them. Delegation and looping are **not** Python code this time -- they are decisions the orchestrator makes itself, following its own instructions:

1. Delegate to `planner_agent` to get a short plan for the goal.
2. Delegate to `react_agent` with the goal and plan, to execute it with tools and produce a draft. If `react_agent` reports that part of the plan failed, the orchestrator is told to delegate back to `planner_agent` for a revised plan covering only the remaining work, then to `react_agent` again.
3. Delegate to `reflect_agent` to review the draft (`GOOD` or `REVISE`).
4. If `REVISE`, delegate back to `react_agent` with the feedback, then to `reflect_agent` again -- the orchestrator's instructions say to do this **at most twice**.
5. Present the final draft once `reflect_agent` says `GOOD`, or the revision limit is told to have been reached.

**Two different kinds of bound are at work below, and it matters which is which:**

- The "at most twice" revision limit in step 4 is **prompt-only** -- it exists as text in `orchestrator_agent`'s instruction, which the model could fail to follow.
- The `max_turns` / `max_tool_calls` / `max_tokens` / `max_seconds` budgets passed to `run_and_show` below are **Python-enforced** -- they stop the run outright no matter what any agent's instructions say, exactly like Part 1's. They are set higher here than in Part 1 because one orchestrator run can involve three specialists and up to two reflect cycles.

The code-enforced budgets are a safety net around the whole thing; the revision count inside that net is only requested, not guaranteed -- Exercise 1 below asks you to check which one actually ends up stopping a run.


In [29]:
planner_agent = Agent(
    name="planner_agent",
    description="Writes a short numbered plan for a scheduling goal. Does not execute anything.",
    model=ADK_MODEL,
    instruction=(
        "Write a short plan (3-5 steps) for how to answer the given scheduling goal using "
        "deadlines, policies, and date math. Format it as a numbered list, one step per "
        "line, like '1. ...'. Only write the plan -- do not answer the request itself. If "
        "you are told that a step already failed, write a revised plan for only the "
        "remaining work instead of repeating completed steps."
    ),
)

react_agent = Agent(
    name="react_agent",
    description="Carries out a plan using the scheduling tools and produces a draft answer.",
    model=ADK_MODEL,
    instruction=(
        "You will be given a goal and either a plan or feedback on a previous draft. Before "
        "every tool call, and before your final answer, write one line starting with "
        "'Thought:' explaining your reasoning -- never skip this. Use the available tools "
        "to carry out the plan -- do not guess or invent values. If a tool reports ok: "
        "false for something the plan needs, do not substitute a made-up value: say clearly "
        "in your final response which part could not be completed and why, so it can be "
        "replanned. Otherwise, produce a draft answer to the goal."
    ),
    tools=[lookup_deadline, lookup_policy, calculate_offset],
)

reflect_agent = Agent(
    name="reflect_agent",
    description="Reviews a draft answer against the goal and answers GOOD or REVISE.",
    model=ADK_MODEL,
    instruction=(
        "Review the draft answer against the goal you are given. Start your reply with "
        "exactly one word, GOOD or REVISE, then in 2-3 sentences say what is missing, "
        "unsupported, or incorrect -- or say why the draft looks good. Answer REVISE if any "
        "required date, deliverable, or policy detail is missing or unsupported."
    ),
)
planner_agent_tool = AgentTool(agent = planner_agent)
react_agent_tool = AgentTool(agent = react_agent)
reflect_agent_tool = AgentTool(agent = reflect_agent)

orchestrator_agent = Agent(
    name="orchestrator_agent",
    model=ADK_MODEL,
    instruction=(
        "You coordinate three specialists to answer a scheduling goal: planner_agent, "
        "react_agent, and reflect_agent. Before each transfer, write one line starting with "
        "'Thought:' explaining why you are transferring to that agent now. Follow this "
        "procedure exactly:\n"
        "1. Transfer to planner_agent with the goal to get a short plan.\n"
        "2. Transfer to react_agent with the goal and the plan to execute it and produce a "
        "draft. If react_agent reports that part of the plan failed, transfer to "
        "planner_agent again with what failed, get a revised plan for only the remaining "
        "work, then transfer to react_agent again with just that remaining work.\n"
        "3. Transfer to reflect_agent with the goal and the current draft.\n"
        "4. If reflect_agent answers REVISE, transfer to react_agent with the goal, the "
        "draft, and the feedback to produce a revised draft, then repeat step 3. Do this at "
        "most twice in total.\n"
        "5. Once reflect_agent answers GOOD, or you have revised twice, present the final "
        "draft to the user and state the final reflection verdict."
    ),
    tools=[planner_agent_tool, react_agent_tool, reflect_agent_tool],
)

goal = "Create a Project 2 presentation brief with a checkpoint seven days before it and list required deliverables."
# Higher than Part 1: this run can involve three specialists and up to two reflect cycles.
_ = await run_and_show(
    orchestrator_agent, goal, label="ORCHESTRATOR",
    max_turns=300, max_tool_calls=100, max_tokens=300000, max_seconds=900,
)



====================== ORCHESTRATOR ======================

[loop 1] [orchestrator_agent] Reasoning:   Thought: I am requesting a short, concrete plan from the planner to structure the project brief and the checkpoint timing.

[loop 2] [orchestrator_agent] Action:      planner_agent({'request': "Create a short plan for: 'Create a Project 2 presentation brief with a checkpoint seven days before it and list required deliverables.'"})
[loop 2] [orchestrator_agent] Observation: {'result': '1. Set the final Project 2 presentation date and apply the policy-based lead time to determine the presentation brief deadline. \n2. Compute the checkpoint date as the presentation date minus seven days; create a calendar entry with reminders and adjust for weekends/holidays per policy. \n3. Define and assign the required deliverables for the brief (e.g., executive summary, problem statement and goals, audience, slide deck outline, final slides, speaker notes, data sources, visuals, risks and mitigation

### Forcing a failure, to see delegated replanning (attempt to) fire

`project_3_report` does not exist in `DEADLINES`, so the lookup will fail. Whether `orchestrator_agent` correctly notices this and delegates back to `planner_agent`, as its instructions say to, is up to the model's own judgment -- not guaranteed by code. That is the trade-off from the markdown above, made concrete. The run is still bounded by the same explicit budgets regardless of how that plays out.


In [30]:
failing_goal = "Look up the deadline for project_3_report and create a checkpoint seven days before it."
_ = await run_and_show(
    orchestrator_agent, failing_goal, label="ORCHESTRATOR (forced failure)",
    max_turns=30, max_tool_calls=10, max_tokens=30_000, max_seconds=90,
)



====================== ORCHESTRATOR (forced failure) ======================

[loop 1] [orchestrator_agent] Reasoning:   Thought: I will consult planning to generate a concise plan for locating the deadline and computing a checkpoint seven days prior.

[loop 2] [orchestrator_agent] Action:      planner_agent({'request': 'Goal: Look up the deadline for project_3_report and create a checkpoint seven days before that deadline. Provide a short plan with steps: identify where the deadline is stored, verify the deadline date, compute the checkpoint date as deadline minus seven days, and outline the action items to set or communicate the checkpoint.'})
[loop 2] [orchestrator_agent] Observation: {'result': '1. Identify where the deadline for project_3_report is stored (project tracker or file) and locate the exact deadline value.  \n2. Verify the deadline date (and time zone) per policy, confirming it against the source and ensuring consistent date format.  \n3. Compute the checkpoint date as 

## Exercise 1: Which bound actually stops a run -- the prompt's, or the code's?

`orchestrator_agent`'s instruction says to revise "at most twice"; separately, `run_and_show` enforces `max_turns`, `max_tool_calls`, `max_tokens`, and `max_seconds` no matter what. These are different kinds of bound, and this exercise asks you to tell them apart empirically.

1. Write a goal that `reflect_agent` is likely to keep marking `REVISE` -- for example, ask about a deliverable or policy that is not in `DEADLINES`/`POLICIES`, or combine several requirements the synthetic tools cannot fully satisfy.
2. Run it through `orchestrator_agent` with `run_and_show`, first with generous budgets (e.g. `max_turns=30, max_tool_calls=10`), then again with much tighter ones (e.g. `max_turns=6, max_tool_calls=2`).
3. Read the `[budget]` summary line at the end of each run. In the generous-budget run, did the orchestrator stop itself after two revisions as instructed, or did a code-enforced budget stop it first anyway? In the tight-budget run, which budget triggered the stop?
4. Report what you observed, and propose one concrete change to `orchestrator_agent`'s instruction or to the code that would make the revision count itself reliably enforced, not just requested.


In [31]:
# EXERCISE 1: Write a goal likely to trigger repeated REVISE verdicts, run it through
# orchestrator_agent with generous and then tight budgets, and report which bound
# (the prompt's "at most twice", or one of run_and_show's code-enforced budgets)
# actually stopped each run, using the printed [budget] summary line as evidence.
stress_test_goal = ""  # fill in


## Exercise 2: Compare the ReAct agent and the orchestrator on the same goal

1. Write one realistic scheduling goal of your own that needs at least two of the three tools (`lookup_deadline`, `lookup_policy`, `calculate_offset`).
2. Run it once through `simple_react_agent` (Part 1) and once through `orchestrator_agent` (Part 2), using the same goal text and comparable budgets for both.
3. For each run, record the `[budget]` summary line (turns, tool calls, tokens, elapsed time) plus: total agent-to-agent delegations (orchestrator only), whether a plan existed that you could read before execution started, and the final answer.
4. Decide and justify: for this specific goal, was the orchestrator's extra structure and delegation worth its added turns, calls, and tokens, or would the simple ReAct agent have produced an equally good answer for less? Connect your answer to the "do not reward activity" principle from this week's material -- more agents and more steps are not automatically better.


In [32]:
# EXERCISE 2: Run the same goal through simple_react_agent and orchestrator_agent with
# comparable budgets, record each [budget] summary plus delegation count and plan
# visibility, and justify which architecture was the better fit for this specific goal.
comparison_goal = ""  # fill in


## Takeaways

- The ReAct agent (Part 1) decides its whole path turn by turn, alone; there is no plan and no delegation.
- The orchestrator (Part 2) does not contain a Python loop at all: `orchestrator_agent` delegates to `planner_agent`, `react_agent`, and `reflect_agent` by name through ADK's `sub_agents` mechanism, and the looping over react-then-reflect-again is the orchestrator's own repeated decision, not a `while` statement.
- Both agents run under four explicit, code-enforced budgets -- turns, tool calls, tokens, and wall-clock time -- passed openly at each call site. These stop a run regardless of what any agent's own instructions say.
- Those budgets are not the same thing as the orchestrator's "at most twice" revision rule, which lives only in its prompt. Exercise 1 is designed to make that difference show up in practice, not just in theory.
- "Reasoning" printed below is each agent's own visible, stated rationale (a `Thought:` line its instruction requires), not a hidden chain-of-thought -- which is exactly the kind of evidence Week 5 says an evaluator needs, and all that an OpenAI reasoning model exposes through this kind of API regardless of framework.
- Neither architecture is strictly better: a Python-orchestrated loop gives you code-enforced bounds and step-level visibility at the cost of writing that control flow by hand; an ADK `sub_agents` orchestrator gives you declarative, low-code composition at the cost of the model being responsible for honoring its own procedure. Which one is appropriate depends on how consequential it is if a prompt-only rule is not actually respected.
